Exploring Household, Person, and Tax Unit Relationships in a Microsimulation
=======================================
One must exercise extreme caution when dealing with variables that are associated with different entities. Using`map_to` without careful thought can lead to unintended consequences. Using `map_to="person"` is safe from an unintended aggregation standpoint. There are some examples below that starts by building a pseudo-relational database and then seeing what we can do with one structure using `map_to`.

In [1]:
import pandas as pd
from policyengine_us import Microsimulation

sim = Microsimulation(dataset="hf://policyengine/policyengine-us-data/enhanced_cps_2024.h5")

hh_person_rel = pd.DataFrame({
    'household_id': sim.calculate('household_id', map_to='person'),
    'person_id': sim.calculate('person_id', map_to='person')
})

tax_unit_person_rel = pd.DataFrame({
    'tax_unit_id': sim.calculate('tax_unit_id', map_to='person'),
    'person_id': sim.calculate('person_id', map_to='person')
})

person = pd.DataFrame({
    'person_id': sim.calculate('person_id', map_to='person'),
    'age': sim.calculate('age', map_to='person')
})

tax_unit = pd.DataFrame({
    'tax_unit_id': sim.calculate('tax_unit_id', map_to='tax_unit'),
    'eitc_child_count': sim.calculate('eitc_child_count', map_to='tax_unit'),
    'tax_unit_is_filer': sim.calculate('tax_unit_is_filer', map_to='tax_unit')
})

/home/baogorek/envs/pe/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tax_unit.loc[tax_unit.tax_unit_id ==  17565102]

,tax_unit_id,eitc_child_count,tax_unit_is_filer
29168,17565102,4,1.0


Now let's get the people associated with that tax unit:

In [3]:
people_in_tax_unit = tax_unit_person_rel.loc[tax_unit_person_rel.tax_unit_id ==  17565102]
people_in_tax_unit

,tax_unit_id,person_id
53247,17565102,17565102
53248,17565102,17565103
53249,17565102,17565104
53250,17565102,17565105
53251,17565102,17565106


Let's look at the households of those people:

In [4]:
households_of_people = hh_person_rel.loc[hh_person_rel.person_id.isin(people_in_tax_unit.person_id)]
households_of_people

,household_id,person_id
53247,175651,17565102
53248,175651,17565103
53249,175651,17565104
53250,175651,17565105
53251,175651,17565106


One household for these 5 people, all in the same tax unit. Now let's look at a person level attribute, the ages of the people:

In [5]:
attributes_of_people = person.loc[person.person_id.isin(people_in_tax_unit.person_id)]
attributes_of_people

,person_id,age
53247,17565102,36.0
53248,17565103,15.0
53249,17565104,14.0
53250,17565105,11.0
53251,17565106,6.0


It looks like 4 children and a working-age parent. Can we do this in one step?

In [6]:
p_df = pd.DataFrame({                                                                                                                                 
    'person_id': sim.calculate('person_id', map_to='person'),                                                                                      
    'tax_unit_id': sim.calculate('tax_unit_id', map_to='person'),                                                                                     
    'household_id': sim.calculate('household_id', map_to='person'),                                                                                   
    'tax_unit_is_filer': sim.calculate('tax_unit_is_filer', map_to='person'),                                                                         
    'eitc_child_count': sim.calculate('eitc_child_count', map_to='person'),                                                                           
    'age': sim.calculate('age', map_to='person')                                                                                                      
})

p_df.loc[p_df.tax_unit_id == 17565102]

,person_id,tax_unit_id,household_id,tax_unit_is_filer,eitc_child_count,age
53247,17565102,17565102,175651,1.0,4,36.0
53248,17565103,17565102,175651,1.0,4,15.0
53249,17565104,17565102,175651,1.0,4,14.0
53250,17565105,17565102,175651,1.0,4,11.0
53251,17565106,17565102,175651,1.0,4,6.0


Yeah that actually looks fine.

Let me look for tax units that span muliple households and households that span multiple tax units

In [7]:
merged_df = pd.merge(tax_unit_person_rel, hh_person_rel, on='person_id')
unique_relations = merged_df[['tax_unit_id', 'household_id']].drop_duplicates()
household_counts = unique_relations.groupby('tax_unit_id')['household_id'].count()
spanning_tax_units = household_counts[household_counts > 1]

spanning_ids_list = spanning_tax_units.index.tolist()
print("\nList of spanning tax_unit_ids:")
print(spanning_ids_list)


List of spanning tax_unit_ids:
[]


Tax Units are completely nested within households. But here's an example that shows that mapping to household can still be dangerous even without person_ids involved:

In [8]:
# First, here's a big household with multiple tax units:
p_df.loc[p_df.household_id == 172866]

,person_id,tax_unit_id,household_id,tax_unit_is_filer,eitc_child_count,age
52459,17286602,17286602,172866,1.0,0,64.0
52460,17286603,17286602,172866,1.0,0,41.0
52461,17286604,17286606,172866,1.0,0,38.0
52462,17286605,17286604,172866,1.0,0,34.0
52463,17286606,17286610,172866,1.0,0,21.0
52464,17286607,17286607,172866,1.0,0,20.0
52465,17286608,17286611,172866,1.0,0,19.0
52466,17286609,17286605,172866,1.0,4,44.0
52467,17286610,17286605,172866,1.0,4,14.0
52468,17286611,17286605,172866,1.0,4,13.0


Now let's see what would have happened if we had tried to go after this tax unit information at a household level:

In [9]:
hh_df = pd.DataFrame({
    'household_id': sim.calculate('household_id', map_to='household'),
    'tax_unit_id': sim.calculate('tax_unit_id', map_to='household'),
    'tax_unit_is_filer': sim.calculate('tax_unit_is_filer', map_to='household'),
    'eitc_child_count': sim.calculate('eitc_child_count', map_to='household')
})
hh_df.loc[hh_df.household_id == 172866]


,household_id,tax_unit_id,tax_unit_is_filer,eitc_child_count
20738,172866,138292859.0,8.0,4.0


There are a couple of things to note:
1. `tax_unit_id` 138292859 does not exist. It is somehow a result of the calculate method.
2. `tax_unit_is_filer` as a defined varialbe boolean, but it is `8.0` here. If you are counting tax_unit_ids, that may be what you want, but there are certainly not 8 tax units with eitc_child_count = 4, there's only 1. You could imagine a cases that are more complicated where there are different numbers of eitc_child_counts over different tax units in the household.

I believe that Tax Unit level information *should only be mapped to the household level with careful attention.* It's dangerous! The same is true for the Person level. Both David and I got bit this week by person ages being summed within household.